# Grammar Scoring Engine for Spoken Audio

**Task:** given a 45–60 s `.wav` recording, predict a continuous grammar score in [0, 5] (MOS Likert rubric).
**Metrics:** RMSE (lower is better) and Pearson correlation (higher is better).

## Report summary

**Idea.** Grammar quality is a property of *what* is said (syntax, completeness of sentences, complexity) and partly of *how* it is said (hesitations, repairs, fluency). So the pipeline combines three views of each recording:

1. **Transcript-based features** – Whisper ASR transcribes the audio. From the transcript we compute surface statistics (length, vocabulary richness, sentence length, filler words, repetitions), ASR confidence (Whisper's average log-probability – disfluent / ungrammatical speech is usually less confidently decoded) and **GPT-2 perplexity** of the text (a well-known proxy for grammatical fluency: ungrammatical text has higher perplexity).
2. **Sentence embedding of the transcript** (MiniLM) – captures semantic/syntactic content.
3. **Acoustic embedding** – mean-pooled Whisper encoder states (two 30 s windows, since Whisper takes 30 s at a time) – captures fluency, pacing, hesitations.

**Models.** Several simple regressors (Ridge, SVR, gradient boosting), one per feature block, trained with 5-fold CV. Their out-of-fold predictions are blended with a non-negative linear stacker. Final predictions are clipped to [0, 5].

**Evaluation.** 5-fold cross-validated RMSE / Pearson on the training set (honest estimate), plus the required **RMSE on the full training data** after the final fit (optimistic by nature – it is reported because the assignment demands it).

> **Setup:** runs locally (VS Code / Jupyter). Needs `ffmpeg` (`brew install ffmpeg`) and internet for the first model download. On a CPU, feature extraction takes roughly 30–90 min with Whisper `base`; it is cached to disk afterwards. Set `QUICK_TEST = True` first to verify the pipeline in a couple of minutes.

## 1. Setup & configuration

In [ ]:
%pip -q install openai-whisper sentence-transformers librosa soundfile

In [ ]:
import os, re, glob, warnings, pickle, random
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # must be set BEFORE importing torch (Mac GPU fallback)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

# ---------------- CONFIG (local machine) ----------------
DATA_ROOT       = "/Users/apple/Downloads/Dataset_Final"   # folder with train.csv, test.csv, train/, test/
TRAIN_AUDIO_DIR = os.path.join(DATA_ROOT, "train")
TEST_AUDIO_DIR  = os.path.join(DATA_ROOT, "test")

WORK_DIR     = "./working"      # features cache + submission.csv are saved here
WHISPER_SIZE = "base"           # CPU-friendly. Use "small" only if you have a GPU / lots of time
QUICK_TEST   = False            # True -> use 30 train / 10 test files just to check that everything runs
CACHE_FILE   = os.path.join(WORK_DIR, f"features_{WHISPER_SIZE}{'_quick' if QUICK_TEST else ''}.pkl")
os.makedirs(WORK_DIR, exist_ok=True)

In [ ]:
train_df   = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
test_df    = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))

if QUICK_TEST:                                   # tiny subset for a fast smoke test
    train_df, test_df = train_df.head(30).copy(), test_df.head(10).copy()

train_df["path"] = train_df.filename.map(lambda f: os.path.join(TRAIN_AUDIO_DIR, f))
test_df["path"]  = test_df.filename.map(lambda f: os.path.join(TEST_AUDIO_DIR, f))

n_miss_tr = (~train_df.path.map(os.path.exists)).sum()
n_miss_te = (~test_df.path.map(os.path.exists)).sum()
print("train:", train_df.shape, "| missing audio:", n_miss_tr)
print("test :", test_df.shape,  "| missing audio:", n_miss_te)
assert n_miss_tr == 0 and n_miss_te == 0, "Audio files not found - check TRAIN_AUDIO_DIR / TEST_AUDIO_DIR"
train_df.head()

## 2. Exploratory analysis

In [ ]:
import soundfile as sf
durs = [sf.info(p).duration for p in train_df.path[:150]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=train_df.label, ax=ax[0], color="steelblue")
ax[0].set_title("Training label distribution"); ax[0].set_xlabel("Grammar score")
ax[1].hist(durs, bins=20, color="darkorange"); ax[1].set_title("Audio duration (s), first 150 files")
plt.tight_layout(); plt.show()
print(train_df.label.describe())
print("Sample rate:", sf.info(train_df.path[0]).samplerate)

**Observations.** Labels are concentrated around 2.5–5 with a tail of a few `0` scores (probably empty / unintelligible recordings) and almost no 1–1.5. Predicting the mean (~3.3) gives an RMSE ≈ 1.24, which is the baseline to beat. Clips are 45–60 s, so a single 30 s Whisper window would discard half of each recording – we therefore process two windows.

## 3. Feature extraction

Audio is loaded at 16 kHz mono (Whisper's native rate). For every file we produce (a) a transcript + ASR confidence, (b) the acoustic embedding. Results are cached to disk so the expensive step runs once.

In [ ]:
import whisper
# openai-whisper cannot be moved to Apple's MPS GPU (sparse-tensor buffer error), so on a Mac:
#   - transcription (decoder) runs on the CPU
#   - the encoder (used for the acoustic embeddings) is copied to the GPU
import copy
ASR_DEVICE = "cpu" if DEVICE == "mps" else DEVICE
asr = whisper.load_model(WHISPER_SIZE, device=ASR_DEVICE)
enc = asr.encoder if ASR_DEVICE == DEVICE else copy.deepcopy(asr.encoder).to(DEVICE).eval()
print("ASR on:", ASR_DEVICE, "| encoder embeddings on:", DEVICE)

def load_audio(path):
    return whisper.load_audio(path)            # float32, 16 kHz, mono (uses ffmpeg)

@torch.no_grad()
def whisper_embed(audio):
    """Mean-pooled Whisper-encoder states over consecutive 30 s windows (max 2 windows)."""
    embs = []
    win = 30 * 16000
    for start in range(0, max(len(audio), 1), win)[:2]:
        chunk = audio[start:start + win]
        if len(chunk) < 2 * 16000:              # ignore tiny tails
            continue
        n_valid = int(np.ceil(len(chunk) / 16000 * 50))        # encoder frames: 50 / second
        mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(chunk)).to(DEVICE)
        if DEVICE == "cuda": mel = mel.half()
        h = enc(mel.unsqueeze(0))[0]                   # (1500, d)
        embs.append(h[:min(n_valid, 1500)].float().mean(0).cpu().numpy())
    return np.mean(embs, 0) if embs else np.zeros(asr.dims.n_audio_state, dtype=np.float32)

def transcribe(audio):
    res = asr.transcribe(audio, language="en", fp16=(DEVICE == "cuda"),
                         temperature=0.0, condition_on_previous_text=False)
    segs = res["segments"]
    return dict(
        text=res["text"].strip(),
        avg_logprob=np.mean([s["avg_logprob"] for s in segs]) if segs else -5.0,
        min_logprob=np.min([s["avg_logprob"] for s in segs]) if segs else -5.0,
        no_speech=np.mean([s["no_speech_prob"] for s in segs]) if segs else 1.0,
        compression=np.mean([s["compression_ratio"] for s in segs]) if segs else 0.0,
        n_segments=len(segs),
        duration=len(audio) / 16000,
    )

In [ ]:
def extract_all(df):
    rows, embs = [], []
    for p in tqdm(df.path, desc="ASR + encoder"):
        audio = load_audio(p)
        rows.append(transcribe(audio)); embs.append(whisper_embed(audio))
    return pd.DataFrame(rows), np.vstack(embs)

if os.path.exists(CACHE_FILE):
    cache = pickle.load(open(CACHE_FILE, "rb"))
    print("Loaded cached features")
else:
    tr_asr, tr_emb = extract_all(train_df)
    te_asr, te_emb = extract_all(test_df)
    cache = dict(tr_asr=tr_asr, tr_emb=tr_emb, te_asr=te_asr, te_emb=te_emb)
    pickle.dump(cache, open(CACHE_FILE, "wb"))
tr_asr, tr_emb, te_asr, te_emb = cache["tr_asr"], cache["tr_emb"], cache["te_asr"], cache["te_emb"]
print(tr_asr.shape, tr_emb.shape, te_asr.shape, te_emb.shape)
tr_asr.head(3)

In [ ]:
# free GPU memory from Whisper before loading the language models
del asr, enc; import gc; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

### 3.1 Text features: surface statistics + GPT-2 perplexity

* **Perplexity** of the transcript under GPT-2 is a proxy for grammatical naturalness (computed overall and per sentence; we keep mean / median / max).
* Surface features: word count, speaking rate, type-token ratio, mean sentence length, filler-word rate (*um, uh, er, like, you know*), immediate word repetitions (repairs/false starts), proportion of long words and of subordinating conjunctions (proxy for complex grammar), and the share of sentences that do not end in a proper verb-like pattern is approximated by very short sentence fraction.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
tok = GPT2TokenizerFast.from_pretrained("gpt2")
lm  = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()

@torch.no_grad()
def ppl(text, max_len=256):
    if len(text.split()) < 3: return np.nan
    ids = tok(text, return_tensors="pt", truncation=True, max_length=max_len).input_ids.to(DEVICE)
    ids = torch.cat([torch.tensor([[tok.bos_token_id]], device=DEVICE), ids], 1)
    loss = lm(ids, labels=ids).loss.item()
    return float(np.exp(min(loss, 15)))

FILLERS = {"um", "uh", "er", "erm", "hmm", "ah", "like", "actually", "basically"}
SUBORD  = {"because", "although", "though", "while", "whereas", "since", "unless", "if", "when", "which", "that", "who", "whose", "whom", "however", "therefore"}

def text_features(text, duration):
    words = re.findall(r"[A-Za-z']+", text.lower())
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip().split()) > 0]
    n = max(len(words), 1)
    sent_lens = [len(s.split()) for s in sents] or [0]
    sent_ppl = [ppl(s) for s in sents if len(s.split()) >= 4]
    sent_ppl = [x for x in sent_ppl if not np.isnan(x)] or [np.nan]
    return dict(
        n_words=len(words), wpm=len(words) / max(duration, 1) * 60,
        ttr=len(set(words)) / n, n_sents=len(sents),
        mean_sent_len=np.mean(sent_lens), max_sent_len=np.max(sent_lens), std_sent_len=np.std(sent_lens),
        short_sent_frac=np.mean([l <= 3 for l in sent_lens]),
        filler_rate=sum(w in FILLERS for w in words) / n,
        repeat_rate=sum(a == b for a, b in zip(words, words[1:])) / n,
        long_word_frac=np.mean([len(w) >= 7 for w in words]) if words else 0,
        subord_rate=sum(w in SUBORD for w in words) / n,
        ppl_all=ppl(text, 400), ppl_mean=np.nanmean(sent_ppl), ppl_med=np.nanmedian(sent_ppl), ppl_max=np.nanmax(sent_ppl),
    )

def build_hand_features(asr_df):
    tf = pd.DataFrame([text_features(t, d) for t, d in tqdm(zip(asr_df.text, asr_df.duration), total=len(asr_df), desc="text feats")])
    out = pd.concat([asr_df.drop(columns=["text"]).reset_index(drop=True), tf], axis=1)
    for c in ["ppl_all", "ppl_mean", "ppl_med", "ppl_max"]:
        out[c] = np.log1p(out[c])            # perplexity is heavy-tailed -> log
    return out

X_hand_tr = build_hand_features(tr_asr)
X_hand_te = build_hand_features(te_asr)
med = X_hand_tr.median()
X_hand_tr, X_hand_te = X_hand_tr.fillna(med), X_hand_te.fillna(med)   # empty transcripts -> median
X_hand_tr.describe().T.head(25)

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
X_txt_tr = st_model.encode(tr_asr.text.tolist(), batch_size=32, show_progress_bar=True)
X_txt_te = st_model.encode(te_asr.text.tolist(), batch_size=32, show_progress_bar=True)
X_aud_tr, X_aud_te = tr_emb, te_emb
y = train_df.label.values.astype(float)
print(X_hand_tr.shape, X_txt_tr.shape, X_aud_tr.shape)

### 3.2 Which hand-crafted features relate to the grammar score?

In [ ]:
corr = X_hand_tr.assign(label=y).corr()["label"].drop("label").sort_values()
plt.figure(figsize=(7, 6)); corr.plot.barh(color=np.where(corr > 0, "seagreen", "indianred"))
plt.title("Pearson correlation of hand-crafted features with grammar score"); plt.xlabel("r"); plt.tight_layout(); plt.show()

Perplexity-type features should correlate **negatively** (more natural text → higher score) while ASR confidence (`avg_logprob`), sentence length and vocabulary richness correlate **positively** – this is what we expect if the features carry grammatical information.

## 4. Modelling

Each feature block gets its own regressor, evaluated with repeated-seed **5-fold CV** (stratified on the rounded label so rare scores appear in every fold). Out-of-fold (OOF) predictions are then blended.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
strat = np.clip(np.round(y), 1, 5).astype(int)   # merge the rare 0/1 scores into one stratum
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(y)), strat))

NPC = lambda k: max(2, min(k, len(y) * 4 // 5 - 2))   # PCA size that also works for the tiny QUICK_TEST subset
models = {
    "Ridge | Whisper-encoder":  (lambda: make_pipeline(StandardScaler(), PCA(NPC(128), random_state=SEED), Ridge(alpha=300)), "aud"),
    "SVR   | Whisper-encoder":  (lambda: make_pipeline(StandardScaler(), PCA(NPC(128), random_state=SEED), SVR(C=3, epsilon=0.2, gamma="scale")), "aud"),
    "Ridge | MiniLM text":      (lambda: make_pipeline(StandardScaler(), Ridge(alpha=500)), "txt"),
    "SVR   | MiniLM text":      (lambda: make_pipeline(StandardScaler(), SVR(C=3, epsilon=0.2)), "txt"),
    "HistGB| hand-crafted":     (lambda: HistGradientBoostingRegressor(max_depth=3, learning_rate=0.04, max_iter=300, l2_regularization=1.0, random_state=SEED), "hand"),
    "Ridge | hand-crafted":     (lambda: make_pipeline(StandardScaler(), Ridge(alpha=30)), "hand"),
    "SVR   | all concatenated": (lambda: make_pipeline(StandardScaler(), PCA(NPC(160), random_state=SEED), SVR(C=3, epsilon=0.2)), "all"),
}
TR = {"aud": X_aud_tr, "txt": X_txt_tr, "hand": X_hand_tr.values,
      "all": np.hstack([X_aud_tr, X_txt_tr, X_hand_tr.values])}
TE = {"aud": X_aud_te, "txt": X_txt_te, "hand": X_hand_te.values,
      "all": np.hstack([X_aud_te, X_txt_te, X_hand_te.values])}

oof = {}
for name, (mk, blk) in models.items():
    p = np.zeros(len(y))
    for tr_i, va_i in folds:
        m = mk().fit(TR[blk][tr_i], y[tr_i]); p[va_i] = m.predict(TR[blk][va_i])
    oof[name] = np.clip(p, 0, 5)
    print(f"{name:26s}  CV RMSE={rmse(y, oof[name]):.3f}  Pearson={pearsonr(y, oof[name])[0]:.3f}")
print(f"{'Baseline (mean)':26s}  CV RMSE={rmse(y, np.full_like(y, y.mean())):.3f}")

### 4.1 Stacking
A non-negative linear regression learns how much to trust each base model (non-negativity keeps the blend stable with only ~770 samples).

In [ ]:
names = list(models)
O = np.column_stack([oof[n] for n in names])
stack = LinearRegression(positive=True).fit(O, y)
print(dict(zip(names, np.round(stack.coef_, 3))), "intercept", round(stack.intercept_, 3))

# honest estimate of the stack: nested CV over the OOF matrix
p_stack = np.zeros(len(y))
for tr_i, va_i in folds:
    s = LinearRegression(positive=True).fit(O[tr_i], y[tr_i]); p_stack[va_i] = s.predict(O[va_i])
p_stack = np.clip(p_stack, 0, 5)
cv_rmse, cv_r = rmse(y, p_stack), pearsonr(y, p_stack)[0]
print(f"\nStacked ensemble  CV RMSE = {cv_rmse:.3f} | CV Pearson = {cv_r:.3f}")

## 5. Evaluation & visualisations (cross-validated)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
# model comparison
res = pd.DataFrame({"RMSE": [rmse(y, oof[n]) for n in names] + [cv_rmse]}, index=names + ["STACK"]).sort_values("RMSE")
res.RMSE.plot.barh(ax=ax[0], color="steelblue"); ax[0].axvline(rmse(y, np.full_like(y, y.mean())), color="red", ls="--", label="mean baseline")
ax[0].set_title("CV RMSE by model"); ax[0].legend()
# predicted vs true
jit = np.random.uniform(-.08, .08, len(y))
ax[1].scatter(y + jit, p_stack, alpha=.5, s=14); ax[1].plot([0, 5], [0, 5], "r--")
ax[1].set_xlabel("True score"); ax[1].set_ylabel("Predicted (OOF)"); ax[1].set_title(f"Stack: RMSE={cv_rmse:.3f}, r={cv_r:.3f}")
# residuals
ax[2].hist(y - p_stack, bins=30, color="seagreen"); ax[2].set_title("Residuals (true - predicted)")
plt.tight_layout(); plt.show()

print("Mean absolute error per true label:")
print(pd.DataFrame({"label": y, "abs_err": np.abs(y - p_stack)}).groupby("label").abs_err.agg(["mean", "count"]).round(3).T)

## 6. Final fit, training-set RMSE (required) and test predictions
All base models are refit on the **full training set**, blended with the stacker, and used to predict the test set.

In [ ]:
final_models = {n: models[n][0]().fit(TR[models[n][1]], y) for n in names}
train_pred_base = np.column_stack([np.clip(final_models[n].predict(TR[models[n][1]]), 0, 5) for n in names])
train_pred = np.clip(stack.predict(train_pred_base), 0, 5)

TRAIN_RMSE = rmse(y, train_pred)
print("=" * 55)
print(f"  TRAINING RMSE (full-fit model, train data) : {TRAIN_RMSE:.4f}")
print(f"  TRAINING Pearson                           : {pearsonr(y, train_pred)[0]:.4f}")
print(f"  5-fold CV RMSE (honest generalisation)     : {cv_rmse:.4f}")
print(f"  5-fold CV Pearson                          : {cv_r:.4f}")
print("=" * 55)

test_base = np.column_stack([np.clip(final_models[n].predict(TE[models[n][1]]), 0, 5) for n in names])
test_pred = np.clip(stack.predict(test_base), 0, 5)

plt.figure(figsize=(6, 4)); sns.kdeplot(y, label="train true", fill=True); sns.kdeplot(test_pred, label="test predicted", fill=True)
plt.legend(); plt.title("Score distributions"); plt.xlim(0, 5); plt.show()

In [ ]:
# submission: same columns as sample_submission.csv, one row per file in test.csv
submission = pd.DataFrame({"filename": test_df.filename, "label": test_pred})
assert list(submission.columns) == list(sample_sub.columns)
submission.to_csv(os.path.join(WORK_DIR, "submission.csv"), index=False)
print(submission.shape); submission.head()

## 7. Discussion

**Pipeline architecture.** `wav (16 kHz)` → Whisper ASR → transcript + confidence → {surface stats, GPT-2 perplexity, MiniLM embedding}; in parallel `wav` → Whisper encoder → pooled acoustic embedding. Each block → its own regressor → non-negative stacker → clip to [0, 5].

**Results.** See the cell in §6: the *training* RMSE is (as always) lower than the cross-validated RMSE, which is the number that predicts leaderboard behaviour. Compare the CV RMSE with the mean-prediction baseline (≈1.24) to see the gain.

**Interpretation.** Perplexity and ASR confidence correlate with the score in the expected direction (§3.2); the acoustic embedding adds fluency information that the text alone cannot give; stacking reduces variance from the individual models.

**Limitations & ideas for improvement**
* ASR often "auto-corrects" ungrammatical speech, hiding errors – a verbatim-style prompt or larger Whisper model helps.
* Labels are sparse at the low end (0–1.5), so low scores are over-predicted (regression to the mean); see the per-label error table.
* Add a grammar-error checker (e.g. LanguageTool) or a fine-tuned CoLA grammaticality classifier as extra features.
* Fine-tune a speech/text transformer end-to-end (e.g. wav2vec2 or DeBERTa on transcripts) with more time/GPU.
* Ensemble several Whisper sizes and seeds.